# NB22 · Iterators & Generators

`advanced/` · notebook 22 of 32 · 38 questions + a mini-project

Every `for` loop you've written has quietly used two built-ins: `iter()` to
get an iterator, and `next()` to pull one item at a time until
`StopIteration` says there's nothing left. This notebook makes that
machinery visible, then shows you how to build your own with iterator
classes and generator functions (`yield`).

Generators are *lazy*: they produce one value when asked and then pause.
That lets you process files bigger than memory, describe infinite
sequences, and chain small steps into pipelines. You already met generator
expressions (NB10) and `itertools` (NB15); now you'll see how they work.

The traps here: an iterator is used up after one pass, a generator's body
doesn't start until the first `next()`, and `tee` quietly stores
everything it has buffered.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB21: dunder methods

## What you'll practise

- Iterable vs iterator; iter() and next(); StopIteration
- Writing an iterator class
- Generator functions, yield, generator state
- yield from; infinite generators
- Lazy pipelines; itertools: tee, takewhile, dropwhile, starmap
- send, throw, close, and a generator's return value

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB22-Q01 · L1 · iter() and next() · Write

Get an iterator over `tickets` with `iter()`, then print the first two tickets using `next()`, one per line.

**Starting code (already in the cell below):**

```python
tickets = ["A12", "B07", "C33"]
```

<details><summary><b>Hint</b></summary>

`iter(some_list)` gives an iterator. Each `next(iterator)` returns the next item.

</details>

<details><summary><b>Expected output</b></summary>

```text
A12
B07
```

</details>

<details><summary><b>Solution</b></summary>

```python
queue = iter(tickets)
print(next(queue))
print(next(queue))
```

An iterator remembers where it is. The first `next()` gives `A12`, the second gives `B07`, and `C33` is still waiting. A `for` loop does exactly this behind the scenes.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import islice

for ticket in islice(tickets, 2):
    print(ticket)
```

When you want "the first n items", `islice` (NB15) says it in one step and works on any iterable, including ones with no length. Calling `next()` by hand is for when you need items one at a time at different places in your code.

</details>

In [ ]:
tickets = ["A12", "B07", "C33"]

# your code here


### NB22-Q02 · L1 · Generator functions · Write

Write a generator function `count_up(n)` that yields `1`, `2`, ... up to `n`. Print `list(count_up(5))`.

<details><summary><b>Hint</b></summary>

A function containing `yield` becomes a generator. Use a loop and `yield` each number instead of collecting them.

</details>

<details><summary><b>Expected output</b></summary>

```text
[1, 2, 3, 4, 5]
```

</details>

<details><summary><b>Solution</b></summary>

```python
def count_up(n):
    number = 1
    while number <= n:
        yield number
        number += 1


print(list(count_up(5)))
```

Each `yield` hands one value to whoever is asking and pauses the function right there. `list()` keeps asking until the function ends, collecting `[1, 2, 3, 4, 5]`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def count_up(n):
    yield from range(1, n + 1)


print(list(count_up(5)))
```

`yield from` (Q14) hands over every item of another iterable. For this exact job `range(1, n + 1)` already exists, so in real code you'd skip the generator entirely.

</details>

In [ ]:
# your code here


### NB22-Q03 · L1 · StopIteration · Error

An iterator over a one-item list. Which error does this raise?

**Which error does this raise?**

```python
batch = iter(["only"])
print(next(batch))
print(next(batch))
```

<details><summary><b>Hint</b></summary>

What can `next()` give back once the items have run out?

</details>

<details><summary><b>Expected output</b></summary>

```text
only
StopIteration
```

</details>

<details><summary><b>Solution</b></summary>

`StopIteration`. It's how an iterator says "I'm finished". A `for` loop catches it for you and ends quietly; a bare `next()` lets it escape.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
batch = iter(["only"])
print(next(batch))
print(next(batch, "empty"))
```

`next(iterator, default)` returns the default instead of raising. Seniors use it whenever running out is a normal outcome, not an error (see Q08).

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB22-Q04 · L1 · Generator functions · Predict

Calling a generator function. Predict the output.

**What does this print?**

```python
def greet():
    yield "hi"
    yield "bye"


gen = greet()
print(type(gen).__name__)
print(list(gen))
```

<details><summary><b>Hint</b></summary>

Calling a function that contains `yield` doesn't run its body. What does it return instead?

</details>

<details><summary><b>Expected output</b></summary>

```text
generator
['hi', 'bye']
```

</details>

<details><summary><b>Solution</b></summary>

`greet()` returns a *generator object* (type name `generator`) without running any of the body. `list(gen)` then runs it to the end, collecting both yielded values.

</details>

<details><summary><b>Senior dev solution</b></summary>

Printing `gen` itself shows something like `<generator object greet at 0x7f...>`, with a memory address that changes every run. Seniors print `type(gen).__name__` or the values instead.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L2 · Easy

### NB22-Q05 · L2 · Iterator classes · Write

Write an iterator class `Countdown(start)` with `__iter__` and `__next__`.
Loop over `Countdown(3)` printing each number, then print `Liftoff!`:

```text
3
2
1
Liftoff!
```

<details><summary><b>Hint</b></summary>

`__iter__` returns `self`. `__next__` returns the current number and moves on, or raises `StopIteration` when it reaches 0.

</details>

<details><summary><b>Expected output</b></summary>

```text
3
2
1
Liftoff!
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


for number in Countdown(3):
    print(number)
print("Liftoff!")
```

That's the whole iterator protocol: `__iter__` returns the iterator and `__next__` returns the next value or raises `StopIteration`. The `for` loop calls `iter()` once, then `next()` until it sees `StopIteration`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def countdown(start):
    while start > 0:
        yield start
        start -= 1


for number in countdown(3):
    print(number)
print("Liftoff!")
```

A generator function writes `__iter__`, `__next__` and `StopIteration` for you, and keeps local variables between calls. Seniors write iterator classes only when the object needs extra methods or state that callers can inspect.

</details>

In [ ]:
# your code here


### NB22-Q06 · L2 · Generator functions · Predict

A generator with `print()` calls inside. Predict the output.

**What does this print?**

```python
def readings():
    print("start")
    yield 1
    print("middle")
    yield 2
    print("end")


gen = readings()
print("created")
print(next(gen))
print("got first")
print(next(gen))
for leftover in gen:
    print(leftover)
```

<details><summary><b>Hint</b></summary>

The body runs only when someone calls `next()`, and only as far as the next `yield`.

</details>

<details><summary><b>Expected output</b></summary>

```text
created
start
1
got first
middle
2
end
```

</details>

<details><summary><b>Solution</b></summary>

Creating the generator runs nothing, so `created` comes first. The first `next()` runs up to the first `yield`: `start`, then `1`. The second runs from there to the second `yield`: `middle`, then `2`. The `for` loop's `next()` runs the rest (`end`), hits the end of the function, and stops without printing a value.

</details>

<details><summary><b>Senior dev solution</b></summary>

This is the whole idea behind lazy pipelines: no work happens until a value is needed. It also means code at the top of a generator (like input checks) runs late. Q37 shows that trap.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB22-Q07 · L2 · Iterable vs iterator · Predict

A list, an iterator and a generator expression. Predict the output.

**What does this print?**

```python
scores = [70, 85, 90]
it = iter(scores)
print(sum(it), sum(it))
evens = (n for n in range(6) if n % 2 == 0)
print(4 in evens, list(evens))
print(sum(scores), sum(scores))
```

<details><summary><b>Hint</b></summary>

An iterator is used up after one pass. `in` on an iterator consumes items until it finds a match.

</details>

<details><summary><b>Expected output</b></summary>

```text
245 0
True []
245 245
```

</details>

<details><summary><b>Solution</b></summary>

The first `sum(it)` uses every item, so the second sees nothing and gives `0`. `4 in evens` pulls `0`, `2`, `4`, finds a match and stops, but those items are gone, and nothing is left for `list()`. A list is not an iterator: each `sum(scores)` starts a fresh pass.

</details>

<details><summary><b>Senior dev solution</b></summary>

When you need several passes, keep the *iterable* (the list) and let each loop make its own iterator. If you were handed an iterator, convert it with `list()` once, if it fits in memory.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB22-Q08 · L2 · iter() and next() · Write

Print the first reading above `30`, then the first reading above `40`.
There is none above 40, so print `None` for that one. Use `next()` with
a default.

```text
31.5
None
```

**Starting code (already in the cell below):**

```python
temps = [22.0, 28.4, 31.5, 35.2, 29.9]
```

<details><summary><b>Hint</b></summary>

A generator expression like `(t for t in temps if t > 30)` produces matches lazily. `next(iterator, None)` returns `None` when it's empty.

</details>

<details><summary><b>Expected output</b></summary>

```text
31.5
None
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(next((t for t in temps if t > 30), None))
print(next((t for t in temps if t > 40), None))
```

The generator expression only checks readings until it finds the first match, so the search stops early. With no match it runs out, and the default `None` comes back instead of `StopIteration`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def first_above(readings, limit):
    return next((t for t in readings if t > limit), None)


print(first_above(temps, 30))
print(first_above(temps, 40))
```

`next(generator, default)` is the standard "find the first match" idiom. Wrapping it in a named function says *what* is being searched for, and the extra brackets around the generator expression are required when it isn't the only argument.

</details>

In [ ]:
temps = [22.0, 28.4, 31.5, 35.2, 29.9]

# your code here


### NB22-Q09 · L2 · Generator functions · Write

Write a generator `running_total(amounts)` that yields the total so far after each amount. Print `list(running_total(sales))`: `[120, 200, 245, 445]`.

**Starting code (already in the cell below):**

```python
sales = [120, 80, 45, 200]
```

<details><summary><b>Hint</b></summary>

Keep a `total` variable inside the generator. It survives between `yield`s.

</details>

<details><summary><b>Expected output</b></summary>

```text
[120, 200, 245, 445]
```

</details>

<details><summary><b>Solution</b></summary>

```python
def running_total(amounts):
    total = 0
    for amount in amounts:
        total += amount
        yield total


print(list(running_total(sales)))
```

A generator's local variables are frozen while it's paused, so `total` keeps its value from one `yield` to the next. That's generator *state*, with no class or global needed.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import accumulate

print(list(accumulate(sales)))
```

`itertools.accumulate` (NB15) is this exact generator, and accepts a function for running max or product: `accumulate(sales, max)`.

</details>

In [ ]:
sales = [120, 80, 45, 200]

# your code here


### NB22-Q10 · L2 · Infinite generators · Write

Write an **infinite** generator `ticket_ids(prefix)` that yields
`T-001`, `T-002`, `T-003`, ... forever. Print the first three IDs, one per
line, using `next()`.

<details><summary><b>Hint</b></summary>

A `while True:` loop with a `yield` inside never ends, which is fine: it only runs when asked. Format the number with `:03d`.

</details>

<details><summary><b>Expected output</b></summary>

```text
T-001
T-002
T-003
```

</details>

<details><summary><b>Solution</b></summary>

```python
def ticket_ids(prefix):
    number = 1
    while True:
        yield f"{prefix}-{number:03d}"
        number += 1


ids = ticket_ids("T")
print(next(ids))
print(next(ids))
print(next(ids))
```

An infinite loop is safe inside a generator because it pauses at every `yield`. Never call `list()` on it, though: that would run forever.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import count, islice


def ticket_ids(prefix):
    return (f"{prefix}-{n:03d}" for n in count(1))


for ticket in islice(ticket_ids("T"), 3):
    print(ticket)
```

`itertools.count(1)` is the infinite counter, and `islice` takes a safe number of items. Infinite generators are common for ID makers, retry delays and polling loops.

</details>

In [ ]:
# your code here


### NB22-Q11 · L2 · Iterable vs iterator · Predict

Which objects are iterators? Predict the output.

**What does this print?**

```python
nums = [1, 2]
print(iter(nums) is iter(nums))
it = iter(nums)
print(iter(it) is it)
print(hasattr(nums, "__next__"), hasattr(it, "__next__"))
```

<details><summary><b>Hint</b></summary>

An *iterable* can hand out a fresh iterator each time you ask. An *iterator* returns itself from `iter()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
False
True
False True
```

</details>

<details><summary><b>Solution</b></summary>

A list is iterable: each `iter(nums)` makes a **new** iterator, so the two are different objects. An iterator's `__iter__` returns itself, so `iter(it) is it`. Only the iterator has `__next__`: a list can't say "next" because it doesn't track a position.

</details>

<details><summary><b>Senior dev solution</b></summary>

That is why you can loop over a list twice but not over an iterator. `isinstance(x, collections.abc.Iterator)` is the clean way to ask "is this an iterator?" in real code.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB22-Q12 · L2 · iter() and next() · Write

Print each word in `words` on its own line **without** a `for` loop. Use
`iter()`, `next()`, a `while True:` loop and `try`/`except
StopIteration`.

**Starting code (already in the cell below):**

```python
words = ["red", "green", "blue"]
```

<details><summary><b>Hint</b></summary>

Get the iterator once, before the loop. Inside the loop, `next()` either gives a word or raises `StopIteration`, which is your signal to `break`.

</details>

<details><summary><b>Expected output</b></summary>

```text
red
green
blue
```

</details>

<details><summary><b>Solution</b></summary>

```python
it = iter(words)
while True:
    try:
        word = next(it)
    except StopIteration:
        break
    print(word)
```

This is what `for word in words:` does for you: call `iter()` once, then `next()` until `StopIteration`. Putting only the `next()` call inside the `try` keeps a `StopIteration` from elsewhere from being mistaken for the end.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
for word in words:
    print(word)
```

Seniors write the `for` loop, of course. Knowing the desugared version is what lets you write iterator classes, read `StopIteration` tracebacks, and pull items by hand when a loop doesn't fit.

</details>

In [ ]:
words = ["red", "green", "blue"]

# your code here


---

## L3 · Intermediate

### NB22-Q13 · L3 · Infinite generators · Write

Write an infinite generator `fib()` that yields the Fibonacci numbers
`0, 1, 1, 2, 3, ...`. Print the first 10 as a list using
`itertools.islice`.

<details><summary><b>Hint</b></summary>

Keep two variables `a, b = 0, 1`. Yield `a`, then move both along with `a, b = b, a + b`.

</details>

<details><summary><b>Expected output</b></summary>

```text
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
```

</details>

<details><summary><b>Solution</b></summary>

```python
from itertools import islice


def fib():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


print(list(islice(fib(), 10)))
```

The generator holds only two numbers at a time, however far you go. `islice(fib(), 10)` asks for exactly ten values and then stops asking, so the infinite loop never runs away.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import islice


def fib():
    """Yield the Fibonacci numbers forever, starting at 0."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


print(list(islice(fib(), 10)))
```

Same code, plus a docstring that says "forever", the most important fact for a caller. Compare NB11: the recursive version needed a cache, while the generator is fast with no memory cost at all.

</details>

In [ ]:
# your code here


### NB22-Q14 · L3 · yield from · Write

Write a generator `all_orders(*batches)` that yields every order from
every batch, in order. Use `yield from`. Print
`list(all_orders(morning, lunch, evening))`.

**Starting code (already in the cell below):**

```python
morning = ["A1", "A2"]
lunch = []
evening = ["B1"]
```

<details><summary><b>Hint</b></summary>

`yield from some_iterable` yields each of its items in turn.

</details>

<details><summary><b>Expected output</b></summary>

```text
['A1', 'A2', 'B1']
```

</details>

<details><summary><b>Solution</b></summary>

```python
def all_orders(*batches):
    for batch in batches:
        yield from batch


print(list(all_orders(morning, lunch, evening)))
```

`yield from batch` is short for `for order in batch: yield order`. Empty batches simply contribute nothing.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import chain

print(list(chain(morning, lunch, evening)))
```

`itertools.chain` is this generator, ready-made. `chain.from_iterable(...)` does the same when the batches themselves come from an iterator. `yield from` earns its keep in recursive generators (Q38) and in passing return values (Q33).

</details>

In [ ]:
morning = ["A1", "A2"]
lunch = []
evening = ["B1"]

# your code here


### NB22-Q15 · L3 · Iterator classes · Write

Write an **iterable** class `Playlist` (not an iterator) whose `__iter__`
is a generator that yields each song. Show it can be looped over twice by
printing `list(p)` two times:

```text
['Intro', 'Outro']
['Intro', 'Outro']
```

<details><summary><b>Hint</b></summary>

If `__iter__` contains `yield`, each call returns a brand-new generator, so each loop starts from the beginning.

</details>

<details><summary><b>Expected output</b></summary>

```text
['Intro', 'Outro']
['Intro', 'Outro']
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __iter__(self):
        for song in self.songs:
            yield song


p = Playlist(["Intro", "Outro"])
print(list(p))
print(list(p))
```

The object itself holds the data, and every `for` loop or `list()` call gets a fresh generator from `__iter__`. That's the difference between an iterable (reusable, like a list) and an iterator (one pass).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __iter__(self):
        return iter(self.songs)

    def __len__(self):
        return len(self.songs)


p = Playlist(["Intro", "Outro"])
print(list(p))
print(list(p))
```

When the data already lives in a list, `return iter(self.songs)` hands out the list's own iterator. Adding `__len__` (NB21) lets `list()` size its result up front and makes `if playlist:` work.

</details>

In [ ]:
# your code here


### NB22-Q16 · L3 · Iterator classes · Fix

The second `list(c)` should also print `[3, 2, 1]`, but it's empty. Fix `Countdown` so it can be looped over any number of times.

**Buggy code (copied into the cell below):**

```python
class Countdown:
    def __init__(self, start):
        self.start = start
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


c = Countdown(3)
print(list(c))
print(list(c))
```

<details><summary><b>Hint</b></summary>

An object that returns `self` from `__iter__` is a one-pass iterator. Make `__iter__` start a fresh count each time instead.

</details>

<details><summary><b>Expected output</b></summary>

```text
[3, 2, 1]
[3, 2, 1]
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        current = self.start
        while current > 0:
            yield current
            current -= 1


c = Countdown(3)
print(list(c))
print(list(c))
```

The original stored its position on the object, and after one pass `current` was `0` for good. Now the position is a local variable inside a generator, and each `__iter__` call gets its own.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return iter(range(self.start, 0, -1))


c = Countdown(3)
print(list(c))
print(list(c))
```

Seniors keep data (`start`) on the object and loop position in the iterator, never mixed together. Then nested loops over the same object also work, which a self-iterator can't do.

</details>

In [ ]:
class Countdown:
    def __init__(self, start):
        self.start = start
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


c = Countdown(3)
print(list(c))
print(list(c))


### NB22-Q17 · L3 · itertools · Write

A sensor sends `-999` when it fails, and readings after a failure can't
be trusted. Print the readings **before** the first failure with
`takewhile`, and the readings from the failure onwards with `dropwhile`:

```text
[21.0, 21.4, 22.1]
[-999, 22.5, 23.0]
```

**Starting code (already in the cell below):**

```python
readings = [21.0, 21.4, 22.1, -999, 22.5, 23.0]
```

<details><summary><b>Hint</b></summary>

Both take a test function and an iterable. `takewhile` yields while the test is true and stops for good at the first false one. `dropwhile` skips while it's true, then yields everything else.

</details>

<details><summary><b>Expected output</b></summary>

```text
[21.0, 21.4, 22.1]
[-999, 22.5, 23.0]
```

</details>

<details><summary><b>Solution</b></summary>

```python
from itertools import takewhile, dropwhile

print(list(takewhile(lambda r: r != -999, readings)))
print(list(dropwhile(lambda r: r != -999, readings)))
```

Unlike a filter, `takewhile` stops at the first failing item and never looks again, even though `22.5` and `23.0` would pass the test. `dropwhile` is its mirror: once the test fails, every remaining item comes through unchecked.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import takewhile, dropwhile

FAILED = -999


def is_valid(reading):
    return reading != FAILED


print(list(takewhile(is_valid, readings)))
print(list(dropwhile(is_valid, readings)))
```

A named sentinel (`FAILED`) and a named test read better than a magic number repeated in two lambdas. Both functions are lazy, so they also work on an endless sensor stream.

</details>

In [ ]:
readings = [21.0, 21.4, 22.1, -999, 22.5, 23.0]

# your code here


### NB22-Q18 · L3 · itertools · Write

`lines` holds `(unit_price, qty)` pairs. Use `itertools.starmap` with
`operator.mul` to print the line totals as a list, then the order total
rounded to 2 decimals:

```text
[9.0, 12.99, 3.0]
24.99
```

**Starting code (already in the cell below):**

```python
lines = [(4.5, 2), (12.99, 1), (0.75, 4)]
```

<details><summary><b>Hint</b></summary>

`starmap(f, pairs)` calls `f(*pair)` for each pair, unpacking it into arguments.

</details>

<details><summary><b>Expected output</b></summary>

```text
[9.0, 12.99, 3.0]
24.99
```

</details>

<details><summary><b>Solution</b></summary>

```python
from itertools import starmap
from operator import mul

totals = list(starmap(mul, lines))
print(totals)
print(round(sum(totals), 2))
```

`map(f, items)` passes each item as one argument; `starmap(f, items)` unpacks each tuple into several arguments, so `mul(4.5, 2)` and so on. The list is kept so it can be printed and summed: a bare `starmap` would be used up after the first pass.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import starmap
from operator import mul

totals = [price * qty for price, qty in lines]
print(totals)
print(round(sum(totals), 2))
```

Many seniors find the comprehension clearer than `starmap(mul, ...)`: it names the parts. `starmap` shines when the function already exists and the data is already in argument tuples, such as `starmap(pow, pairs)`.

</details>

In [ ]:
lines = [(4.5, 2), (12.99, 1), (0.75, 4)]

# your code here


### NB22-Q19 · L3 · Lazy evaluation · Predict

Generator expressions and the data they read. Predict the output.

**What does this print?**

```python
nums = [1, 2, 3]
tens = (n * 10 for n in nums)
nums.append(4)
print(list(tens))

words = ["a", "b"]
upper = (w.upper() for w in words)
words = ["x", "y", "z"]
print(list(upper))
```

<details><summary><b>Hint</b></summary>

A generator expression grabs the *object* it loops over when it's created, but doesn't read any items until asked.

</details>

<details><summary><b>Expected output</b></summary>

```text
[10, 20, 30, 40]
['A', 'B']
```

</details>

<details><summary><b>Solution</b></summary>

`tens` holds the same list object as `nums`. It reads lazily, so the `4` appended later is included: `[10, 20, 30, 40]`. In the second case `words = [...]` makes the name point to a **new** list, but `upper` already holds the old one, so it gives `['A', 'B']`.

</details>

<details><summary><b>Senior dev solution</b></summary>

Lazy code sees the data as it is *when it runs*, not when it was written. Seniors avoid changing a collection while a generator over it is still alive. It's the same aliasing lesson as NB18, with a time delay.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB22-Q20 · L3 · iter() and next() · Error

Calling `next()` on a list. Which error does this raise?

**Which error does this raise?**

```python
colours = ["red", "green"]
print(next(colours))
```

<details><summary><b>Hint</b></summary>

Is a list an iterator? Does it have a `__next__` method?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError: 'list' object is not an iterator`. A list is iterable but not an iterator (Q11). You need `next(iter(colours))`, which gives `red`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
colours = ["red", "green"]
print(next(iter(colours)))
```

For a list, `colours[0]` is simpler. `next(iter(x))` is the idiom for getting the first item from something that can't be indexed, such as a set, a dict's keys or a generator.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB22-Q21 · L3 · Generator functions · Refactor

`even_squares` builds a whole list before returning anything. Turn it into
a generator function with `yield`. Keep the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def even_squares(limit):
    result = []
    for n in range(limit):
        if n % 2 == 0:
            result.append(n * n)
    return result


for square in even_squares(10):
    print(square, end=" ")
print()
```

<details><summary><b>Hint</b></summary>

Replace `result.append(...)` with `yield ...`, and delete the list and the `return`.

</details>

<details><summary><b>Expected output</b></summary>

```text
0 4 16 36 64 
```

</details>

<details><summary><b>Solution</b></summary>

```python
def even_squares(limit):
    for n in range(limit):
        if n % 2 == 0:
            yield n * n


for square in even_squares(10):
    print(square, end=" ")
print()
```

The caller only loops, so it never needed a list. The generator version uses constant memory and starts producing values immediately, which matters when `limit` is ten million.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def even_squares(limit):
    return (n * n for n in range(0, limit, 2))


print(*even_squares(10), end=" \n")
```

A one-line transformation can return a generator expression. `range` with a step of 2 skips the odd numbers instead of testing them. Return a list only when callers need `len()`, indexing or several passes.

</details>

In [ ]:
def even_squares(limit):
    result = []
    for n in range(limit):
        if n % 2 == 0:
            result.append(n * n)
    return result


for square in even_squares(10):
    print(square, end=" ")
print()


### NB22-Q22 · L3 · Generator functions · Refactor

`chunks` builds a list of every batch up front. Make it a generator that
yields one batch at a time. Keep the output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
def chunks(items, size):
    result = []
    i = 0
    while i < len(items):
        result.append(items[i:i + size])
        i += size
    return result


for batch in chunks(["a", "b", "c", "d", "e"], 2):
    print(batch)
```

<details><summary><b>Hint</b></summary>

`range(0, len(items), size)` gives the start of each batch. Yield the slice.

</details>

<details><summary><b>Expected output</b></summary>

```text
['a', 'b']
['c', 'd']
['e']
```

</details>

<details><summary><b>Solution</b></summary>

```python
def chunks(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]


for batch in chunks(["a", "b", "c", "d", "e"], 2):
    print(batch)
```

Each batch is produced only when the loop asks for it. `range` with a step replaces the manual `i` counter, so there's no `while` loop to get wrong.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import batched

for batch in batched(["a", "b", "c", "d", "e"], 2):
    print(list(batch))
```

`itertools.batched` (3.12, NB15) does this and also works on iterators with no length, like file lines. It yields tuples, hence `list(batch)` to keep the output identical.

</details>

In [ ]:
def chunks(items, size):
    result = []
    i = 0
    while i < len(items):
        result.append(items[i:i + size])
        i += size
    return result


for batch in chunks(["a", "b", "c", "d", "e"], 2):
    print(batch)


### NB22-Q23 · L3 · Generator functions · Write

A meter is read once a day. Write a generator `deltas(values)` that yields
how much each reading went up since the one before. Print
`list(deltas(meter))`: `[4, 7, 0, 9]`.

**Starting code (already in the cell below):**

```python
meter = [100, 104, 111, 111, 120]
```

<details><summary><b>Hint</b></summary>

Remember the previous reading in a local variable. The first reading has nothing before it, so it produces no output.

</details>

<details><summary><b>Expected output</b></summary>

```text
[4, 7, 0, 9]
```

</details>

<details><summary><b>Solution</b></summary>

```python
def deltas(values):
    previous = None
    for value in values:
        if previous is not None:
            yield value - previous
        previous = value


print(list(deltas(meter)))
```

`previous` is the generator's memory between `yield`s. Using `None` as a "no reading yet" marker handles the first value without special-casing index 0, so it works on any iterable, not only lists.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import pairwise

print([later - earlier for earlier, later in pairwise(meter)])
```

`itertools.pairwise` (3.10, NB15) yields overlapping pairs `(100, 104), (104, 111), ...`, which is the shape most "compare with the previous item" problems want.

</details>

In [ ]:
meter = [100, 104, 111, 111, 120]

# your code here


---

## L4 · Somewhat Difficult

### NB22-Q24 · L4 · Lazy pipelines · Write

`log` behaves like an open file. Build a lazy pipeline of three generator
functions:

1. `read_lines(f)`: yields each line stripped, skipping blank lines
2. `parse(lines)`: yields `(path, ms)` tuples from lines like `GET /home 120`
3. `slow(records, limit)`: yields only records with `ms >= limit`

Then loop over `slow(parse(read_lines(log)), 500)` **once** and print:

```text
slow requests: 3
total time: 2950 ms
slowest: /search
```

**Starting code (already in the cell below):**

```python
import io

log = io.StringIO("""GET /home 120
POST /cart 950

GET /item 40
GET /search 1300
POST /pay 700
""")
```

<details><summary><b>Hint</b></summary>

Each stage takes an iterable and yields. In the final loop, keep a count, a running total and the slowest record seen so far.

</details>

<details><summary><b>Expected output</b></summary>

```text
slow requests: 3
total time: 2950 ms
slowest: /search
```

</details>

<details><summary><b>Solution</b></summary>

```python
def read_lines(f):
    for line in f:
        line = line.strip()
        if line:
            yield line


def parse(lines):
    for line in lines:
        method, path, ms = line.split()
        yield path, int(ms)


def slow(records, limit):
    for path, ms in records:
        if ms >= limit:
            yield path, ms


count = 0
total = 0
slowest = None
for path, ms in slow(parse(read_lines(log)), 500):
    count += 1
    total += ms
    if slowest is None or ms > slowest[1]:
        slowest = (path, ms)
print("slow requests:", count)
print("total time:", total, "ms")
print("slowest:", slowest[0])
```

Nothing runs until the final `for` asks for a value. Then one line flows through all three stages before the next line is even read, so memory use stays the same whether the log has six lines or six million.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def read_lines(f):
    return (line.strip() for line in f if line.strip())


def parse(lines):
    for line in lines:
        _method, path, ms = line.split()
        yield path, int(ms)


def slow(records, limit):
    return ((path, ms) for path, ms in records if ms >= limit)


slow_requests = list(slow(parse(read_lines(log)), 500))
total = sum(ms for _, ms in slow_requests)
slowest_path, _ = max(slow_requests, key=lambda record: record[1])
print("slow requests:", len(slow_requests))
print("total time:", total, "ms")
print("slowest:", slowest_path)
```

The senior keeps the stages lazy but collects the *filtered* result, which is small, so `len`, `sum` and `max` read clearly. That's a judgement call: stay lazy while data is big, materialise once it's small. `_method` signals "unpacked but unused".

</details>

In [ ]:
import io

log = io.StringIO("""GET /home 120
POST /cart 950

GET /item 40
GET /search 1300
POST /pay 700
""")

# your code here


### NB22-Q25 · L4 · itertools · Write

`stream` is an iterator: you can only read it once. Use `itertools.tee`
to get two independent copies, then print the highest price and the
average price rounded to 2 decimals:

```text
12.99 6.25
```

**Starting code (already in the cell below):**

```python
stream = iter([4.5, 12.99, 0.75, 6.0, 7.0])
```

<details><summary><b>Hint</b></summary>

`a, b = tee(stream)` gives two iterators. Use one for `max()` and the other to add up and count.

</details>

<details><summary><b>Expected output</b></summary>

```text
12.99 6.25
```

</details>

<details><summary><b>Solution</b></summary>

```python
from itertools import tee

for_max, for_avg = tee(stream)
highest = max(for_max)
total = 0
count = 0
for price in for_avg:
    total += price
    count += 1
print(highest, round(total / count, 2))
```

`tee` makes two iterators that each see every item. To do that it must store each item until **both** copies have read it. Here `max()` reads all of `for_max` first, so `tee` ends up holding the entire stream in memory for `for_avg`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
prices = list(stream)
print(max(prices), round(sum(prices) / len(prices), 2))
```

This is the `tee` trap: when one copy runs far ahead of the other, `tee` buffers everything, and a plain `list()` is simpler and faster. Seniors use `tee` only when the copies advance roughly together, like comparing each item with the next.

</details>

In [ ]:
stream = iter([4.5, 12.99, 0.75, 6.0, 7.0])

# your code here


### NB22-Q26 · L4 · itertools · Predict

`tee` and the original iterator. Predict the output.

**What does this print?**

```python
from itertools import tee

it = iter([1, 2, 3, 4])
a, b = tee(it)
print(next(a), next(a))
print(next(it))
print(list(a), list(b))
```

<details><summary><b>Hint</b></summary>

`tee` pulls items from `it` only when one of its copies needs a new one. What happens when you use `it` directly behind `tee`'s back?

</details>

<details><summary><b>Expected output</b></summary>

```text
1 2
3
[4] [1, 2, 4]
```

</details>

<details><summary><b>Solution</b></summary>

`next(a)` twice makes `tee` pull `1` and `2` from `it`, keeping them for `b`. Then `next(it)` takes `3` directly, and `tee` never sees it. So `a` gets only `4`, and `b` gets the buffered `1, 2` plus `4`.

</details>

<details><summary><b>Senior dev solution</b></summary>

After calling `tee(it)`, never touch `it` again. Seniors write `a, b = tee(it)` and immediately forget `it`, or reuse the name: `it, copy = tee(it)`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB22-Q27 · L4 · send, throw and close · Write

Write a generator `ticket_counter(start)` that yields `start`, then
`start + 1`, and so on. If someone calls `.send(n)`, it jumps to `n`
and yields that instead. Run this script:

```py
counter = ticket_counter(1)
print(next(counter), next(counter))
print(counter.send(100))
print(next(counter))
```

It should print `1 2`, then `100`, then `101`.

<details><summary><b>Hint</b></summary>

`received = yield value` gives back whatever was sent in, or `None` after a plain `next()`. Use that to decide the next number.

</details>

<details><summary><b>Expected output</b></summary>

```text
1 2
100
101
```

</details>

<details><summary><b>Solution</b></summary>

```python
def ticket_counter(start):
    number = start
    while True:
        received = yield number
        if received is None:
            number += 1
        else:
            number = received


counter = ticket_counter(1)
print(next(counter), next(counter))
print(counter.send(100))
print(next(counter))
```

`yield` works both ways. It sends `number` out, pauses, and when resumed it evaluates to the value passed to `send()` (`next()` is the same as `send(None)`). `send()` also runs on to the next `yield` and returns that value, which is why it prints `100`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class TicketCounter:
    def __init__(self, start):
        self.next_number = start

    def take(self):
        number = self.next_number
        self.next_number += 1
        return number

    def reset(self, number):
        self.next_number = number


counter = TicketCounter(1)
print(counter.take(), counter.take())
counter.reset(100)
print(counter.take())
print(counter.take())
```

`send()` is worth understanding because `async`/`await` (NB27) is built on it. For everyday state like this, though, most seniors write a small class with named methods: `reset(100)` says more than `send(100)`.

</details>

In [ ]:
# your code here


### NB22-Q28 · L4 · send, throw and close · Error

Sending a value into a generator that hasn't started. Which error does this raise?

**Which error does this raise?**

```python
def echo():
    while True:
        received = yield
        print("got", received)


gen = echo()
gen.send("hello")
```

<details><summary><b>Hint</b></summary>

A new generator is paused *before* its first line, not at a `yield`. Is there a `yield` waiting to receive the value?

</details>

<details><summary><b>Expected output</b></summary>

```text
TypeError
```

</details>

<details><summary><b>Solution</b></summary>

`TypeError: can't send non-None value to a just-started generator`. The value has nowhere to go until the generator has run to its first `yield`. You must *prime* it first with `next(gen)` (or `gen.send(None)`).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def echo():
    while True:
        received = yield
        print("got", received)


gen = echo()
next(gen)
gen.send("hello")
```

Forgetting to prime is the classic coroutine bug. Seniors either prime right after creating the generator (Q34) or wrap creation in a helper that does it, so callers can't forget.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB22-Q29 · L4 · send, throw and close · Write

Write a generator `read_records(lines)` that prints `open` when it starts,
yields each line in upper case, and prints `cleanup` when it finishes **or
is closed early** (use `try`/`finally`). Run this script:

```py
records = read_records(["a", "b", "c"])
print(next(records))
print(next(records))
records.close()
print("closed")
```

```text
open
A
B
cleanup
closed
```

<details><summary><b>Hint</b></summary>

`close()` raises `GeneratorExit` at the paused `yield`. A `finally` block runs on the way out, however the generator ends.

</details>

<details><summary><b>Expected output</b></summary>

```text
open
A
B
cleanup
closed
```

</details>

<details><summary><b>Solution</b></summary>

```python
def read_records(lines):
    print("open")
    try:
        for line in lines:
            yield line.upper()
    finally:
        print("cleanup")


records = read_records(["a", "b", "c"])
print(next(records))
print(next(records))
records.close()
print("closed")
```

The caller stopped after two items, so the loop never finished. `close()` throws `GeneratorExit` into the generator at the paused `yield`, the `finally` block runs, and the generator ends. Without `close()`, cleanup would wait until the generator was garbage collected.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def read_records(lines):
    print("open")
    try:
        for line in lines:
            yield line.upper()
    finally:
        print("cleanup")


records = read_records(["a", "b", "c"])
try:
    print(next(records))
    print(next(records))
finally:
    records.close()
print("closed")
```

A generator that holds a resource (a file, a connection) should clean up in `finally`, and the caller should guarantee `close()`. `contextlib.closing` (NB24) turns that into a `with` block.

</details>

In [ ]:
# your code here


### NB22-Q30 · L4 · send, throw and close · Write

Write a generator `scaled(readings)` that yields each reading plus an
`offset` (starting at `0`). If someone calls
`gen.throw(Recalibrate(5))`, the generator catches it, sets `offset` to
`5`, and carries on with the next reading. Run this script:

```py
gen = scaled([10, 20, 30, 40])
print(next(gen))
print(gen.throw(Recalibrate(5)))
print(next(gen))
print(next(gen))
```

It should print `10`, `25`, `35`, `45`.

**Starting code (already in the cell below):**

```python
class Recalibrate(Exception):
    pass
```

<details><summary><b>Hint</b></summary>

`throw()` raises the exception *at the paused `yield`*. Wrap the `yield` in `try`/`except Recalibrate as e:` and read the number from `e.args[0]`.

</details>

<details><summary><b>Expected output</b></summary>

```text
10
25
35
45
```

</details>

<details><summary><b>Solution</b></summary>

```python
def scaled(readings):
    offset = 0
    for reading in readings:
        try:
            yield reading + offset
        except Recalibrate as e:
            offset = e.args[0]


gen = scaled([10, 20, 30, 40])
print(next(gen))
print(gen.throw(Recalibrate(5)))
print(next(gen))
print(next(gen))
```

The generator is paused at `yield 10` when `throw()` arrives, so the exception appears right there. It's caught, `offset` becomes `5`, and the loop moves on to `20`, yielding `25`. Like `send()`, `throw()` returns the next yielded value.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def scaled(readings):
    offset = 0
    for reading in readings:
        try:
            yield reading + offset
        except Recalibrate as signal:
            (offset,) = signal.args


gen = scaled([10, 20, 30, 40])
print(next(gen))
print(gen.throw(Recalibrate(5)))
print(next(gen))
print(next(gen))
```

`(offset,) = signal.args` fails loudly if the signal carries the wrong number of values. In practice `throw()` is rare in application code; you'll mostly meet it inside frameworks, like `contextlib` (NB24) and `asyncio` (NB27) cancelling a task.

</details>

In [ ]:
class Recalibrate(Exception):
    pass

# your code here


### NB22-Q31 · L4 · Iterable vs iterator · Fix

This should print the average, `4.0`, but it crashes. Fix it.

**Buggy code (copied into the cell below):**

```python
def parse(lines):
    for line in lines:
        yield float(line)


values = parse(["2.5", "3.5", "6.0"])
total = sum(values)
count = len(list(values))
print(total / count)
```

<details><summary><b>Hint</b></summary>

`sum(values)` reads the generator to the end. What's left for the second pass?

</details>

<details><summary><b>Expected output</b></summary>

```text
4.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
def parse(lines):
    for line in lines:
        yield float(line)


values = list(parse(["2.5", "3.5", "6.0"]))
total = sum(values)
count = len(values)
print(total / count)
```

The original raises `ZeroDivisionError`: `sum()` used up the generator, so `list(values)` was empty and `count` was `0`. Converting to a list once makes the values reusable.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def parse(lines):
    for line in lines:
        yield float(line)


total = count = 0
for value in parse(["2.5", "3.5", "6.0"]):
    total += value
    count += 1
print(total / count)
```

When the data might not fit in memory, seniors compute everything in one pass instead of building a list. For averages specifically, `statistics.fmean()` (NB15) also reads an iterator only once.

</details>

In [ ]:
def parse(lines):
    for line in lines:
        yield float(line)


values = parse(["2.5", "3.5", "6.0"])
total = sum(values)
count = len(list(values))
print(total / count)


### NB22-Q32 · L4 · Lazy pipelines · Refactor

This builds three full lists, one per step. Rewrite it as a lazy pipeline
of generator expressions, so no intermediate list is ever built. Keep the
output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
lines = ["5", "x", "12", "", "7", "40"]
cleaned = [line.strip() for line in lines]
numbers = []
for line in cleaned:
    if line.isdigit():
        numbers.append(int(line))
big = [n for n in numbers if n > 6]
print(sum(big))
```

<details><summary><b>Hint</b></summary>

Swap each `[...]` for `(...)`, and turn the loop with `append` into a generator expression too.

</details>

<details><summary><b>Expected output</b></summary>

```text
59
```

</details>

<details><summary><b>Solution</b></summary>

```python
lines = ["5", "x", "12", "", "7", "40"]
cleaned = (line.strip() for line in lines)
numbers = (int(line) for line in cleaned if line.isdigit())
big = (n for n in numbers if n > 6)
print(sum(big))
```

Each generator expression pulls from the one before it. `sum()` drives the whole chain, and each line travels all the way through before the next is touched. The names still document every step.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def parse_ints(lines):
    for line in lines:
        line = line.strip()
        if line.isdigit():
            yield int(line)


lines = ["5", "x", "12", "", "7", "40"]
print(sum(n for n in parse_ints(lines) if n > 6))
```

Once a step has real logic (cleaning *and* validating), seniors give it a named generator function that can be tested on its own. The final filter is simple enough to stay inline.

</details>

In [ ]:
lines = ["5", "x", "12", "", "7", "40"]
cleaned = [line.strip() for line in lines]
numbers = []
for line in cleaned:
    if line.isdigit():
        numbers.append(int(line))
big = [n for n in numbers if n > 6]
print(sum(big))


### NB22-Q33 · L4 · yield from · Predict

A generator that `return`s a value. Predict the output.

**What does this print?**

```python
def inner():
    yield 1
    yield 2
    return "inner done"


def outer():
    result = yield from inner()
    print("outer got:", result)
    yield 3


print(list(outer()))

gen = inner()
next(gen)
next(gen)
try:
    next(gen)
except StopIteration as stop:
    print(stop.value)
```

<details><summary><b>Hint</b></summary>

A generator's `return value` is carried by the `StopIteration` it raises. `yield from` catches that and gives you the value.

</details>

<details><summary><b>Expected output</b></summary>

```text
outer got: inner done
[1, 2, 3]
inner done
```

</details>

<details><summary><b>Solution</b></summary>

`yield from inner()` passes `1` and `2` through to `list()`, then evaluates to `inner`'s return value. The `print` runs while `list()` is still collecting, so `outer got: inner done` appears **before** the list `[1, 2, 3]`. Without `yield from`, the value rides on `StopIteration.value`.

</details>

<details><summary><b>Senior dev solution</b></summary>

A plain `for` loop throws the return value away. That's why return values are rare in ordinary generators and common in coroutine code, where `yield from` (and later `await`, NB27) passes results back up.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L5 · Hard

### NB22-Q34 · L5 · send, throw and close · Write

Write a coroutine `averager()`: each value sent in returns the running
average so far. Prime it, send `10`, `20` and `60`, and print each
result, then close it:

```text
10.0
15.0
30.0
```

<details><summary><b>Hint</b></summary>

Keep `total` and `count` in the generator. The first `yield` has nothing to report yet, so `average` can start as `None`. Prime with `next()` before the first `send()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
10.0
15.0
30.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
def averager():
    total = 0
    count = 0
    average = None
    while True:
        value = yield average
        total += value
        count += 1
        average = total / count


avg = averager()
next(avg)
print(avg.send(10))
print(avg.send(20))
print(avg.send(60))
avg.close()
```

Priming runs to the first `yield`, which hands out `None`. Each `send(v)` resumes with `value = v`, updates the totals, loops round and pauses at `yield average`, so `send()` returns the new average. The state lives in the paused generator, not in globals.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def averager():
    total = 0.0
    count = 0
    average = None
    while True:
        value = yield average
        if value is None:
            continue
        total += value
        count += 1
        average = total / count


def started(coroutine):
    next(coroutine)
    return coroutine


avg = started(averager())
for value in (10, 20, 60):
    print(avg.send(value))
avg.close()
```

`started()` makes priming impossible to forget (Q28), and ignoring `None` means a stray `next()` doesn't crash with `TypeError`. Decorators (NB23) turn `started` into `@coroutine`. Today a class with an `add()` method is usually clearer, but this pattern is how async code began.

</details>

In [ ]:
# your code here


### NB22-Q35 · L5 · iter() and next() · Write

Write a generator `round_robin(*playlists)` that takes one song from each
playlist in turn, skipping playlists once they run out, until all are
empty. Print `list(round_robin(rock, jazz, pop))`:

```text
['R1', 'J1', 'P1', 'R2', 'P2', 'R3', 'P3', 'P4']
```

**Starting code (already in the cell below):**

```python
rock = ["R1", "R2", "R3"]
jazz = ["J1"]
pop = ["P1", "P2", "P3", "P4"]
```

<details><summary><b>Hint</b></summary>

Keep a `deque` of *iterators*, one per playlist. Take one from the left, call `next()` on it: if it gives a song, yield it and put the iterator back on the right; if it raises `StopIteration`, drop it.

</details>

<details><summary><b>Expected output</b></summary>

```text
['R1', 'J1', 'P1', 'R2', 'P2', 'R3', 'P3', 'P4']
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import deque


def round_robin(*playlists):
    queue = deque(iter(p) for p in playlists)
    while queue:
        it = queue.popleft()
        try:
            song = next(it)
        except StopIteration:
            continue
        yield song
        queue.append(it)


print(list(round_robin(rock, jazz, pop)))
```

Each playlist gets its own iterator, which remembers where that playlist is up to. The deque rotates through them. When one raises `StopIteration`, it simply isn't put back, so finished playlists drop out and the rest carry on.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from itertools import chain, zip_longest

_GAP = object()


def round_robin(*playlists):
    rounds = zip_longest(*playlists, fillvalue=_GAP)
    return (song for song in chain.from_iterable(rounds) if song is not _GAP)


print(list(round_robin(rock, jazz, pop)))
```

`zip_longest` takes one from each, round by round, padding finished playlists with a filler. A private sentinel `object()` (NB18) is the safe filler because it can never equal a real song, unlike `None` or `""`.

</details>

In [ ]:
rock = ["R1", "R2", "R3"]
jazz = ["J1"]
pop = ["P1", "P2", "P3", "P4"]

# your code here


### NB22-Q36 · L5 · Infinite generators · Write

`sensor()` is an infinite stream. Write a generator
`moving_average(values, size)` that yields the average of the last `size`
values, starting only once it has `size` values. Print the first five
averages for `size=3`, rounded to 2 decimals:

```text
[3.67, 4.0, 4.33, 4.67, 5.0]
```

**Starting code (already in the cell below):**

```python
def sensor():
    n = 0
    while True:
        yield n * 7 % 10
        n += 1
```

<details><summary><b>Hint</b></summary>

A `deque(maxlen=size)` (NB15) drops the oldest value by itself. Use `islice` to take five results, or the program never ends.

</details>

<details><summary><b>Expected output</b></summary>

```text
[3.67, 4.0, 4.33, 4.67, 5.0]
```

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import deque
from itertools import islice


def moving_average(values, size):
    window = deque(maxlen=size)
    for value in values:
        window.append(value)
        if len(window) == size:
            yield sum(window) / size


averages = islice(moving_average(sensor(), 3), 5)
print([round(a, 2) for a in averages])
```

The sensor gives `0, 7, 4, 1, 8, 5, 2, ...`. The window fills after three values (`(0 + 7 + 4) / 3`) and then slides one value at a time. Every stage is lazy, so the infinite source is fine: `islice` stops asking after five averages.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import deque
from itertools import islice


def moving_average(values, size):
    window = deque(maxlen=size)
    total = 0
    for value in values:
        if len(window) == size:
            total -= window[0]
        window.append(value)
        total += value
        if len(window) == size:
            yield total / size


averages = islice(moving_average(sensor(), 3), 5)
print([round(a, 2) for a in averages])
```

The senior keeps a running total, subtracting the value about to fall out of the window, so each step costs the same however big `size` is. With floats, a long-running total slowly gathers rounding error; for money, use `Decimal` (NB16).

</details>

In [ ]:
def sensor():
    n = 0
    while True:
        yield n * 7 % 10
        n += 1

# your code here


### NB22-Q37 · L5 · Generator functions · Fix

`batches` should reject a size of `0` **when it's called**, so the
`except` below catches it. Instead the error escapes later. Fix it so the
script prints:

```text
bad size: size must be positive
done
```

**Buggy code (copied into the cell below):**

```python
def batches(items, size):
    if size <= 0:
        raise ValueError("size must be positive")
    for start in range(0, len(items), size):
        yield items[start:start + size]


try:
    groups = batches([1, 2, 3], 0)
except ValueError as e:
    print("bad size:", e)
    groups = []
for group in groups:
    print(group)
print("done")
```

<details><summary><b>Hint</b></summary>

Because the function contains `yield`, *none* of its body runs at the call, including the check. Split it into an ordinary function that checks, and a generator that does the work.

</details>

<details><summary><b>Expected output</b></summary>

```text
bad size: size must be positive
done
```

</details>

<details><summary><b>Solution</b></summary>

```python
def batches(items, size):
    if size <= 0:
        raise ValueError("size must be positive")

    def generate():
        for start in range(0, len(items), size):
            yield items[start:start + size]

    return generate()


try:
    groups = batches([1, 2, 3], 0)
except ValueError as e:
    print("bad size:", e)
    groups = []
for group in groups:
    print(group)
print("done")
```

The original creates a generator without running anything, so the `try` succeeds and the `ValueError` only fires in the `for` loop, outside the `try`. Now `batches` has no `yield` of its own: it checks immediately, then returns the inner generator.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def _batches(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]


def batches(items, size):
    if size <= 0:
        raise ValueError("size must be positive")
    return _batches(items, size)


try:
    groups = batches([1, 2, 3], 0)
except ValueError as e:
    print("bad size:", e)
    groups = []
for group in groups:
    print(group)
print("done")
```

A private module-level `_batches` is easier to test than a nested function. The "eager check, lazy work" split is a standard pattern: `itertools.batched(items, 0)` raises at the call for the same reason.

</details>

In [ ]:
def batches(items, size):
    if size <= 0:
        raise ValueError("size must be positive")
    for start in range(0, len(items), size):
        yield items[start:start + size]


try:
    groups = batches([1, 2, 3], 0)
except ValueError as e:
    print("bad size:", e)
    groups = []
for group in groups:
    print(group)
print("done")


### NB22-Q38 · L5 · yield from · Write

`tree` is a folder: dict values are either file sizes (ints) or
sub-folders (dicts). Write a **recursive** generator `walk(folder,
prefix="")` that yields `(path, size)` for every file. Print each file,
then the total size:

```text
docs/cv.pdf 120
docs/notes/todo.txt 3
docs/notes/ideas.txt 5
photo.jpg 900
total 1028
```

**Starting code (already in the cell below):**

```python
tree = {
    "docs": {"cv.pdf": 120, "notes": {"todo.txt": 3, "ideas.txt": 5}},
    "photo.jpg": 900,
}
```

<details><summary><b>Hint</b></summary>

For each entry, build its path. If the value is a dict, `yield from` a recursive `walk` of it with the longer prefix; otherwise yield the file.

</details>

<details><summary><b>Expected output</b></summary>

```text
docs/cv.pdf 120
docs/notes/todo.txt 3
docs/notes/ideas.txt 5
photo.jpg 900
total 1028
```

</details>

<details><summary><b>Solution</b></summary>

```python
def walk(folder, prefix=""):
    for name, value in folder.items():
        path = prefix + name
        if isinstance(value, dict):
            yield from walk(value, path + "/")
        else:
            yield path, value


total = 0
for path, size in walk(tree):
    print(path, size)
    total += size
print("total", total)
```

`yield from walk(...)` passes every file from the sub-folder straight up to the caller, however deep it is. Each level adds its folder name to `prefix`. Total size is counted in the same pass, so the tree is walked only once.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from pathlib import PurePosixPath


def walk(folder, prefix=PurePosixPath()):
    for name, value in folder.items():
        path = prefix / name
        if isinstance(value, dict):
            yield from walk(value, path)
        else:
            yield path, value


total = 0
for path, size in walk(tree):
    print(path, size)
    total += size
print("total", total)
```

`PurePosixPath` (NB13) joins with `/` and never touches the real disk. A Path object as the default is safe because it's immutable, unlike a list default (NB07). `os.walk` and `Path.walk()` are the real-disk versions of this generator.

</details>

In [ ]:
tree = {
    "docs": {"cv.pdf": 120, "notes": {"todo.txt": 3, "ideas.txt": 5}},
    "photo.jpg": 900,
}

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB22-P · L5 · Mini-project: Streaming CSV Transformer · Write

`sales.csv` might be huge, so the transformer must handle **one row at a
time** and never build a list of rows. Build these generator stages:

1. `read_rows(path)`: opens the file and yields each row as a dict
   (`csv.DictReader`)
2. `clean(rows)`: yields `{"region", "product", "total"}` dicts, where
   `total = qty * unit_price`. Bad rows are skipped with a message
   using the file line number (the header is line 1):
   `skipped line N: missing unit_price` when `unit_price` is empty,
   otherwise `skipped line N: bad qty 'two'` when `qty` isn't a whole
   number
3. A function `write_and_total(records, out_path)` that writes each
   record to `out_path` (columns `region,product,total`, totals with 2
   decimals) **as it arrives**, while adding up totals per region. It
   returns `(rows_written, totals)`.

Print the messages, the row count, the output file, and the region
totals sorted by name. Call `tmp.cleanup()` at the end.

```text
skipped line 4: bad qty 'two'
skipped line 6: missing unit_price
wrote 4 rows
region,product,total
North,Tea,9.00
South,Cake,12.99
North,Mug,30.00
East,Tea,4.50
totals: East=4.50 North=39.00 South=12.99
```

**Starting code (already in the cell below):**

```python
import csv
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
in_path = Path(tmp.name) / "sales.csv"
out_path = Path(tmp.name) / "totals.csv"
in_path.write_text(
    "date,region,product,qty,unit_price\n"
    "2026-10-01,North,Tea,2,4.50\n"
    "2026-10-01,South,Cake,1,12.99\n"
    "2026-10-02,North,Mug,two,7.50\n"
    "2026-10-02,North,Mug,4,7.50\n"
    "2026-10-03,East,Coffee,3,\n"
    "2026-10-03,East,Tea,1,4.50\n",
    encoding="utf-8",
)
```

<details><summary><b>Hint</b></summary>

Put the `with open(...)` *inside* `read_rows` and `yield from` the reader, so the file stays open only while rows are flowing. `enumerate(rows, start=2)` gives file line numbers. The skip messages appear while `write_and_total` is pulling rows through, before you print the count.

</details>

<details><summary><b>Expected output</b></summary>

```text
skipped line 4: bad qty 'two'
skipped line 6: missing unit_price
wrote 4 rows
region,product,total
North,Tea,9.00
South,Cake,12.99
North,Mug,30.00
East,Tea,4.50
totals: East=4.50 North=39.00 South=12.99
```

</details>

<details><summary><b>Solution</b></summary>

```python
def read_rows(path):
    with open(path, newline="", encoding="utf-8") as f:
        yield from csv.DictReader(f)


def clean(rows):
    for line_no, row in enumerate(rows, start=2):
        if not row["unit_price"]:
            print(f"skipped line {line_no}: missing unit_price")
            continue
        if not row["qty"].isdigit():
            print(f"skipped line {line_no}: bad qty {row['qty']!r}")
            continue
        total = int(row["qty"]) * float(row["unit_price"])
        yield {"region": row["region"], "product": row["product"], "total": total}


def write_and_total(records, out_path):
    totals = {}
    written = 0
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["region", "product", "total"])
        writer.writeheader()
        for record in records:
            writer.writerow({**record, "total": f"{record['total']:.2f}"})
            region = record["region"]
            totals[region] = totals.get(region, 0) + record["total"]
            written += 1
    return written, totals


written, totals = write_and_total(clean(read_rows(in_path)), out_path)
print(f"wrote {written} rows")
print(out_path.read_text(encoding="utf-8"), end="")
summary = " ".join(f"{region}={amount:.2f}" for region, amount in sorted(totals.items()))
print("totals:", summary)
tmp.cleanup()
```

`read_rows` and `clean` are generators, so building `clean(read_rows(in_path))` reads nothing. `write_and_total` drives the pipeline: each row is read, cleaned, written and counted before the next is read, so memory use doesn't grow with the file. That's also why the skip messages appear before `wrote 4 rows`. The only thing kept is the small `totals` dict.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import defaultdict
from decimal import Decimal

FIELDS = ["region", "product", "total"]


def read_rows(path):
    with open(path, newline="", encoding="utf-8") as f:
        yield from csv.DictReader(f)


def clean(rows):
    for line_no, row in enumerate(rows, start=2):
        if not row["unit_price"]:
            print(f"skipped line {line_no}: missing unit_price")
        elif not row["qty"].isdigit():
            print(f"skipped line {line_no}: bad qty {row['qty']!r}")
        else:
            total = int(row["qty"]) * Decimal(row["unit_price"])
            yield {"region": row["region"], "product": row["product"], "total": total}


def write_and_total(records, out_path):
    totals = defaultdict(Decimal)
    written = 0
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS, lineterminator="\n")
        writer.writeheader()
        for record in records:
            writer.writerow(record | {"total": f"{record['total']:.2f}"})
            totals[record["region"]] += record["total"]
            written += 1
    return written, dict(totals)


try:
    written, totals = write_and_total(clean(read_rows(in_path)), out_path)
    print(f"wrote {written} rows")
    print(out_path.read_text(encoding="utf-8"), end="")
    summary = " ".join(f"{region}={amount:.2f}" for region, amount in sorted(totals.items()))
    print(f"totals: {summary}")
finally:
    tmp.cleanup()
```

`Decimal` (NB16) keeps money exact, and `defaultdict(Decimal)` (NB15) starts each region at `Decimal(0)`. `lineterminator="\n"` stops `csv` writing Windows `\r\n` endings. `try`/`finally` cleans up even if a stage crashes. In production, skip messages would go to `logging` (NB28) rather than `print`.

</details>

In [ ]:
import csv
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
in_path = Path(tmp.name) / "sales.csv"
out_path = Path(tmp.name) / "totals.csv"
in_path.write_text(
    "date,region,product,qty,unit_price\n"
    "2026-10-01,North,Tea,2,4.50\n"
    "2026-10-01,South,Cake,1,12.99\n"
    "2026-10-02,North,Mug,two,7.50\n"
    "2026-10-02,North,Mug,4,7.50\n"
    "2026-10-03,East,Coffee,3,\n"
    "2026-10-03,East,Tea,1,4.50\n",
    encoding="utf-8",
)

# your code here


---

## Done

Next up: **NB23 · Decorators** in `advanced/`.